In [ ]:
# --- COSC6385 environment fix (Colab Nov 2026 image) ---
# The shipped flax is mismatched with the pinned jax on the TPU/GPU image:
#   flax 0.11.x import fails (missing jax._src.core.mutable_array),
#   flax 0.10.6 imports but nnx calls the removed jax.core.get_opaque_trace_state.
# Fix: pin flax 0.10.6 WITHOUT touching jax/jaxlib (protects libtpu), then re-add
# the removed attribute as a shim BEFORE flax is imported.
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','flax==0.10.6'], check=False)
import jax
from jax.extend.core import get_opaque_trace_state as _gots
jax.core.get_opaque_trace_state = _gots
print('env fix applied: flax pinned 0.10.6 + get_opaque_trace_state shim')


# Colab GPU vs. TPU — Starter Notebook (Parts 1–2)

Run this notebook **top to bottom once on each accelerator** you test
(`Runtime → Change runtime type`): at minimum **T4 GPU** and **v5e-1 TPU**; add L4 / A100 / H100 / v6e-1 if you have access.

Each run appends rows to `results_matmul.csv` and `results_cnn.csv`.
**Download both CSVs after every run** — Colab deletes local files when the runtime is recycled.
The final section merges all CSVs you upload and plots the comparison.

The same JAX code runs unchanged on CPU, GPU and TPU. That is the point: differences you measure come from the hardware and XLA, not from different code.

## 0. Environment

In [ ]:
# Flax and Optax are usually preinstalled on Colab. This only installs them if missing
# and does NOT upgrade JAX (upgrading JAX on a TPU runtime can break the libtpu pairing).
import importlib.util, subprocess, sys
for pkg in ("flax", "optax"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
import os, time, datetime, platform, functools
import numpy as np
import pandas as pd
import jax, jax.numpy as jnp
import flax, optax
from flax import nnx

QUICK = os.environ.get("BENCH_QUICK", "0") == "1"   # tiny sizes, for smoke-testing only

dev = jax.devices()[0]
PLATFORM = dev.platform                 # 'cpu' | 'gpu' | 'tpu'
DEVICE_KIND = dev.device_kind           # e.g. 'Tesla T4', 'TPU v5 lite'
RUN_ID = f"{DEVICE_KIND.replace(' ', '_')}_{datetime.datetime.now():%Y%m%d_%H%M%S}"

ENV = dict(
    run_id=RUN_ID, platform=PLATFORM, device_kind=DEVICE_KIND,
    n_devices=jax.device_count(),
    jax=jax.__version__, flax=flax.__version__, optax=optax.__version__,
    python=platform.python_version(),
    colab_runtime=os.environ.get("COLAB_RELEASE_TAG", "n/a"),
)
for k, v in ENV.items():
    print(f"{k:>14}: {v}")
# Note: Colab TPUs today are single-chip (v5e-1, v6e-1), so n_devices is 1.

### Approximate peak throughput
Dense (non-sparse) vendor figures in TFLOPS. **Check these against the vendor datasheet for your report** and cite what you use.
`None` means "fill in yourself" — `% of peak` is left blank for that device.

In [ ]:
PEAK_TFLOPS = {            # key: lowercase substring of device_kind
    #  name           bf16/fp16 tensor   fp32 (non-tensor)
    "t4":           {"bf16": 65.0,  "fp32": 8.1},   # T4 has fp16 tensor cores, no native bf16
    "l4":           {"bf16": 121.0, "fp32": 30.3},
    "a100":         {"bf16": 312.0, "fp32": 19.5},
    "h100":         {"bf16": 989.0, "fp32": 67.0},  # SXM figures
    "tpu v5 lite":  {"bf16": 197.0, "fp32": None},  # v5e
    "tpu v6 lite":  {"bf16": 918.0, "fp32": None},  # v6e (Trillium)
}
def peak_for(kind, dtype_key):
    k = kind.lower()
    for name, d in PEAK_TFLOPS.items():
        if name in k:
            return d.get(dtype_key)
    return None
print("peak bf16:", peak_for(DEVICE_KIND, "bf16"), " peak fp32:", peak_for(DEVICE_KIND, "fp32"))

In [ ]:
def append_csv(rows, path):
    df = pd.DataFrame(rows)
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    return df

def time_call(f, *args, iters=10):
    # First call = trace + XLA compile + run. Timed separately and excluded from the steady state.
    t0 = time.perf_counter(); jax.block_until_ready(f(*args)); first = time.perf_counter() - t0
    jax.block_until_ready(f(*args))                      # one extra warm-up
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); jax.block_until_ready(f(*args)); ts.append(time.perf_counter() - t0)
    return first, float(np.median(ts)), float(np.std(ts))

## 1. Matrix-multiply microbenchmark

Replaces the old "quick benchmark" script. We sweep matrix size `n` and three numeric modes:

| mode | inputs | what the hardware actually does |
|---|---|---|
| `fp32-highest` | float32 | full-precision fp32 (on TPU: several bf16 passes on the MXU) |
| `fp32-default` | float32 | the backend's default precision (TPU: a single bf16 pass; Ampere+/Hopper GPUs: may use TF32) |
| `bf16` | bfloat16 | native low precision on TPU / A100 / H100 / L4; **emulated or slow on T4** |

### 1a. Pitfall demo: asynchronous dispatch

In [ ]:
n = 512 if QUICK else 4096
a = jax.random.normal(jax.random.key(0), (n, n), jnp.float32)
mm = jax.jit(lambda x, y: x @ y)
jax.block_until_ready(mm(a, a))                         # compile first

t0 = time.perf_counter(); out = mm(a, a); t_async = time.perf_counter() - t0
jax.block_until_ready(out)
t0 = time.perf_counter(); jax.block_until_ready(mm(a, a)); t_sync = time.perf_counter() - t0
print(f"without block_until_ready: {t_async*1e3:8.3f} ms   (measures dispatch only)")
print(f"with    block_until_ready: {t_sync*1e3:8.3f} ms   (measures the actual work)")

### 1b. Sweep

In [ ]:
SIZES = [256, 512] if QUICK else [1024, 2048, 4096, 8192]
MODES = {
    "fp32-highest": (jnp.float32,  jax.lax.Precision.HIGHEST, "fp32"),
    "fp32-default": (jnp.float32,  jax.lax.Precision.DEFAULT, "fp32"),
    "bf16":         (jnp.bfloat16, jax.lax.Precision.DEFAULT, "bf16"),
}

@functools.partial(jax.jit, static_argnames="precision")
def matmul(x, y, precision):
    return jnp.matmul(x, y, precision=precision)

rows = []
for n in SIZES:
    ka, kb = jax.random.split(jax.random.key(n))
    for mode, (dt, prec, peak_key) in MODES.items():
        x = jax.random.normal(ka, (n, n), jnp.float32).astype(dt)
        y = jax.random.normal(kb, (n, n), jnp.float32).astype(dt)
        first, med, sd = time_call(functools.partial(matmul, precision=prec), x, y)
        tflops = 2 * n**3 / med / 1e12
        peak = peak_for(DEVICE_KIND, peak_key)
        rows.append(dict(**ENV, n=n, mode=mode, first_call_s=first, median_s=med, std_s=sd,
                         tflops=tflops, pct_peak=(100 * tflops / peak) if peak else np.nan))
        print(f"n={n:5d}  {mode:13s}  first={first*1e3:8.1f} ms  median={med*1e3:8.3f} ms  "
              f"{tflops:7.2f} TFLOPS  {rows[-1]['pct_peak']:5.1f}% of peak")
        del x, y

mm_df = append_csv(rows, "results_matmul.csv")

**Questions for your report (Part 1)**
1. How large must `n` be before each device approaches its steady-state throughput? Why are small matmuls inefficient?
2. Compare `first_call_s` with `median_s`. What is being measured in the first call?
3. On the TPU, why do `fp32-default` and `bf16` perform similarly while `fp32-highest` is slower? Check the numerical difference between the three outputs for one `n`.
4. What fraction of peak did each device reach? Relate this to the roofline/arithmetic-intensity discussion in Wang et al. (2019).

## 2. CNN training comparison (CIFAR-10, Flax NNX)

A small 3-block CNN. The full dataset is placed on the accelerator once and batches are gathered **inside** the jitted step,
so the timing measures the accelerator, not Colab's CPU input pipeline. (Say this in your report — it is a deliberate methodological choice.)

In [ ]:
# TF-free CIFAR-10 loader (keras.datasets needs tensorflow, absent on this image).
import os, urllib.request, tarfile, pickle, numpy as np
def load_cifar10(n_train=None):
    root='/content/cifar-10-batches-py'
    if not os.path.isdir(root):
        tgz='/content/c10.tgz'
        if not os.path.exists(tgz):
            urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', tgz)
        tarfile.open(tgz).extractall('/content')
    def load(f):
        d=pickle.load(open(root+'/'+f,'rb'),encoding='bytes'); return d[b'data'], d[b'labels']
    xs=[];ys=[]
    for i in range(1,6):
        x,y=load('data_batch_%d'%i); xs.append(x); ys+=y
    xtr=np.concatenate(xs).reshape(-1,3,32,32).transpose(0,2,3,1).astype(np.uint8)
    ytr=np.array(ys,np.int32)
    xt,yt=load('test_batch')
    xte=np.array(xt).reshape(-1,3,32,32).transpose(0,2,3,1).astype(np.uint8)
    yte=np.array(yt,np.int32)
    src='cifar10'
    if n_train:
        xtr,ytr,xte,yte = xtr[:n_train],ytr[:n_train],xte[:n_train//4],yte[:n_train//4]
    return xtr,ytr,xte,yte,src

xtr, ytr, xte, yte, DATA_SRC = load_cifar10(n_train=2048 if QUICK else None)
print(DATA_SRC, xtr.shape, xte.shape)
x_train, y_train = jax.device_put(xtr), jax.device_put(ytr)
x_test, y_test = jax.device_put(xte), jax.device_put(yte)


In [ ]:
def pool2(x):                                    # 2x2 max-pool, NHWC
    b, h, w, c = x.shape
    return x.reshape(b, h // 2, 2, w // 2, 2, c).max(axis=(2, 4))

class CNN(nnx.Module):
    def __init__(self, *, dtype, rngs: nnx.Rngs):
        kw = dict(dtype=dtype, param_dtype=jnp.float32, rngs=rngs)   # bf16 compute, fp32 master weights
        self.c1 = nnx.Conv(3, 32, (3, 3), **kw)
        self.c2 = nnx.Conv(32, 64, (3, 3), **kw)
        self.c3 = nnx.Conv(64, 128, (3, 3), **kw)
        self.f1 = nnx.Linear(4 * 4 * 128, 256, **kw)
        self.f2 = nnx.Linear(256, 10, **kw)
        self.dtype = dtype

    def __call__(self, x):
        x = x.astype(self.dtype)
        x = pool2(nnx.relu(self.c1(x)))
        x = pool2(nnx.relu(self.c2(x)))
        x = pool2(nnx.relu(self.c3(x)))
        x = nnx.relu(self.f1(x.reshape(x.shape[0], -1)))
        return self.f2(x).astype(jnp.float32)

def make_trainer(dtype, lr=1e-3):
    model = CNN(dtype=dtype, rngs=nnx.Rngs(0))
    graphdef, params = nnx.split(model)
    tx = optax.adam(lr)

    def loss_fn(params, xb, yb):
        logits = nnx.merge(graphdef, params)(xb)
        return optax.softmax_cross_entropy_with_integer_labels(logits, yb).mean()

    @jax.jit
    def train_step(params, opt_state, x_all, y_all, idx):
        xb = x_all[idx].astype(jnp.float32) / 255.0
        yb = y_all[idx]
        loss, grads = jax.value_and_grad(loss_fn)(params, xb, yb)
        updates, opt_state = tx.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), opt_state, loss

    @jax.jit
    def accuracy(params, x, y):
        logits = nnx.merge(graphdef, params)(x.astype(jnp.float32) / 255.0)
        return (logits.argmax(-1) == y).mean()

    return params, tx.init(params), train_step, accuracy

In [ ]:
# Fill in from Colab's "Resources" panel (compute units per hour) to get cost per epoch.
# Leave None if unknown; the cost column will be blank.
CU_PER_HOUR = None
USD_PER_CU = 0.10            # pay-as-you-go list price at time of writing; verify

EPOCHS = 2 if QUICK else 5
BATCH_SIZES = [128] if QUICK else [128, 512]
DTYPES = {"fp32": jnp.float32, "bf16": jnp.bfloat16}

rows = []
n = x_train.shape[0]
for dname, dt in DTYPES.items():
    for bs in BATCH_SIZES:
        params, opt_state, train_step, accuracy = make_trainer(dt)
        steps = n // bs
        rng = np.random.default_rng(0)
        epoch_times, compile_s = [], None
        for ep in range(EPOCHS):
            perm = rng.permutation(n)[: steps * bs].reshape(steps, bs)
            t0 = time.perf_counter()
            for s in range(steps):
                if ep == 0 and s == 0:
                    tc = time.perf_counter()
                    params, opt_state, loss = train_step(params, opt_state, x_train, y_train, perm[s])
                    jax.block_until_ready(loss); compile_s = time.perf_counter() - tc
                else:
                    params, opt_state, loss = train_step(params, opt_state, x_train, y_train, perm[s])
            jax.block_until_ready(loss)
            epoch_times.append(time.perf_counter() - t0)
        steady = epoch_times[1:] if len(epoch_times) > 1 else epoch_times   # drop epoch 0 (compile)
        ep_s = float(np.median(steady))
        acc = float(accuracy(params, x_test, y_test))
        cost = (ep_s / 3600) * CU_PER_HOUR * USD_PER_CU if CU_PER_HOUR else np.nan
        rows.append(dict(**ENV, data=DATA_SRC, dtype=dname, batch=bs, epochs=EPOCHS,
                         compile_s=compile_s, epoch_s=ep_s, step_ms=1e3 * ep_s / steps,
                         img_per_s=steps * bs / ep_s, test_acc=acc, final_loss=float(loss),
                         usd_per_epoch=cost))
        r = rows[-1]
        print(f"{dname} bs={bs:4d}  compile={compile_s:6.1f}s  epoch={ep_s:7.2f}s  "
              f"{r['img_per_s']:9.0f} img/s  acc={acc:.3f}")

cnn_df = append_csv(rows, "results_cnn.csv")

**Questions for your report (Part 2)**
1. Rank the devices by images/s and by cost per epoch. Do the rankings agree?
2. How does batch size change throughput on each device, and why does the TPU respond more strongly?
3. Does bf16 change test accuracy after the same number of epochs? Does it change speed on the T4?
4. Your matmul results reached some % of peak. Does the CNN reach more or less? What limits it?

## 3. Combine runs and plot
Upload the CSVs from your other runs (Files panel, left sidebar) — name them e.g. `results_matmul_T4.csv`. Every file matching the pattern is merged.

In [ ]:
import glob
import matplotlib.pyplot as plt

def load_all(prefix):
    frames = [pd.read_csv(p) for p in sorted(glob.glob(f"{prefix}*.csv"))]
    return pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()

mm_all, cnn_all = load_all("results_matmul"), load_all("results_cnn")
print(len(mm_all), "matmul rows,", len(cnn_all), "cnn rows from",
      mm_all["device_kind"].nunique() if len(mm_all) else 0, "device(s)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (kind, mode), g in mm_all.groupby(["device_kind", "mode"]):
    g = g.groupby("n", as_index=False)["tflops"].median()
    axes[0].plot(g["n"], g["tflops"], marker="o", label=f"{kind} · {mode}")
axes[0].set(xscale="log", yscale="log", xlabel="matrix size n", ylabel="achieved TFLOPS",
            title="Matmul throughput")
axes[0].legend(fontsize=7)

if len(cnn_all):
    c = cnn_all.groupby(["device_kind", "dtype", "batch"], as_index=False)["img_per_s"].median()
    c["label"] = c["device_kind"] + "\n" + c["dtype"] + " bs" + c["batch"].astype(str)
    axes[1].barh(c["label"], c["img_per_s"])
    axes[1].set(xlabel="images / second", title="CNN training throughput")
    axes[1].tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.savefig("comparison.png", dpi=150); plt.show()

In [ ]:
# Download this run's results (Colab only).
try:
    from google.colab import files
    for p in ("results_matmul.csv", "results_cnn.csv", "comparison.png"):
        if os.path.exists(p):
            files.download(p)
except ImportError:
    print("Not on Colab; files are in", os.getcwd())